In [1]:
import os
import math
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from shapely.geometry import Point

# ==== PATH ====
profile_dir = "data/test_set/output/generator/profiles/whole"
finder_path = "data/test_set/output/finder/finder_method.csv"
output_dir = "data/test_set/output/figures/test_set/cross_sections"
os.makedirs(output_dir, exist_ok=True)

# 🟨 Enter the IDs of the cones for which you wish to create graphs here..
# If the list is empty -> it will create cone_id for all entries present in finder_method.csv
selected_cone_ids = ["55", "38", "14", "22", "18"]   # e.g. ["13", "57"]; or [] for all cones

finder_df = pd.read_csv(finder_path, sep=";")

# optional geometry – only useful if you want to calculate something further
finder_df["geometry"] = finder_df.apply(lambda r: Point(r["x_geo"], r["y_geo"]), axis=1)

# orientation as a number (e.g. "90deg" -> 90.0)
def parse_deg(s: str) -> float:
    # I assume the format "90deg", "5deg", etc.
    return float(str(s).replace("deg", ""))

finder_df["angle_deg"] = finder_df["orientation"].apply(parse_deg)

# map: transect_id -> angle, cone_id
transects_info = (finder_df
                  .groupby("transect_id")
                  .agg({
                      "angle_deg": "first",
                      "cone_id": "first"
                  })
                  .reset_index())

if not selected_cone_ids:
    selected_cone_ids = sorted(transects_info["cone_id"].unique().astype(str))


def attach_distance_to_finder_points(profile_df: pd.DataFrame,
                                     points_df: pd.DataFrame) -> pd.DataFrame:
    """
    For points (x_geo, y_geo) from the finder, it finds the nearest point on the profile
    and assigns it the value of the 'distance' column.
    """
    if points_df.empty:
        return points_df

    dx = profile_df["x_geo"].values[None, :] - points_df["x_geo"].values[:, None]
    dy = profile_df["y_geo"].values[None, :] - points_df["y_geo"].values[:, None]
    dist_sq = dx**2 + dy**2
    nearest_idx = dist_sq.argmin(axis=1)

    points_df = points_df.copy()
    points_df["distance"] = profile_df["distance"].values[nearest_idx]
    return points_df


def plot_axis_pair(cone_id: str,
                   transect_a: str,
                   transect_b: str,
                   angle_axis: float):
    """
    Creates a plot with two profiles superimposed (angle and angle+180°)
    for one cone and one axis + calculates Wco, Wcr, H_left/right, D_left/right.
    """

    def load_profile(tr_id: str):
        prof_path = os.path.join(profile_dir, f"profile_{tr_id}.csv")
        prof = pd.read_csv(prof_path, sep=";")
        return prof

    prof_a = load_profile(transect_a)
    prof_b = load_profile(transect_b)

    pts_a = finder_df[finder_df["transect_id"] == transect_a].copy()
    pts_b = finder_df[finder_df["transect_id"] == transect_b].copy()

    pts_a = attach_distance_to_finder_points(prof_a, pts_a)
    pts_b = attach_distance_to_finder_points(prof_b, pts_b)

    center_a = pts_a[pts_a["type"] == "C"]
    center_b = pts_b[pts_b["type"] == "C"]

    if center_a.empty or center_b.empty:
        print(f"[cone {cone_id}, axis {angle_axis}°] no point C in one of the profiles – I am omitting it.")
        return

    ca_dist = float(center_a["distance"].iloc[0])
    ca_elev = float(center_a["elevation"].iloc[0])
    cb_dist = float(center_b["distance"].iloc[0])

    # --- recalculate the X axis so that C is in the same place and profile B is mirrored ---
    x_min = prof_a["distance"].min()
    x_max = prof_a["distance"].max()
    center_target = 0.5 * (x_min + x_max)

    prof_a_x = prof_a["distance"] - ca_dist + center_target
    prof_b_x = cb_dist - prof_b["distance"] + center_target

    def transform_x(dist_series, center_dist, mirror=False):
        if mirror:
            return center_dist - dist_series + center_target
        else:
            return dist_series - center_dist + center_target

    pts_a["x_plot"] = transform_x(pts_a["distance"], ca_dist, mirror=False)
    pts_b["x_plot"] = transform_x(pts_b["distance"], cb_dist, mirror=True)

    all_pts = pd.concat([pts_a, pts_b], ignore_index=True)

    # --- CENTER (we take C from profile A) ---
    C_x = float(pts_a.loc[pts_a["type"] == "C", "x_plot"].iloc[0])
    C_z = ca_elev

    # --- TOP: we take the extreme left and right ---
    tops = all_pts[all_pts["type"].str.contains("_top", na=False)].copy()
    tops = tops.sort_values("x_plot")
    top_left = top_right = None
    tops_to_plot = []

    if len(tops) >= 2:
        top_left = tops.iloc[0]
        top_right = tops.iloc[-1]
        tops_to_plot = [top_left, top_right]
    elif len(tops) == 1:
        top_left = tops.iloc[0]
        tops_to_plot = [top_left]

    # --- BOTTOM: separately left/right side relative to C_x ---
    bottoms = all_pts[all_pts["type"].str.contains("_bottom", na=False)].copy()

    bottoms_near = []
    bottoms_far = []

    bottom_left_near = bottom_right_near = None

    if not bottoms.empty:
        # left side
        left = bottoms[bottoms["x_plot"] < C_x].copy()
        if not left.empty:
            left_sorted = left.iloc[(left["x_plot"] - C_x).abs().argsort()]
            bottom_left_near = left_sorted.iloc[0]
            bottoms_near.append(bottom_left_near.to_dict())
            if len(left_sorted) > 1:
                bottoms_far.append(left_sorted.iloc[1].to_dict())

        # right side
        right = bottoms[bottoms["x_plot"] >= C_x].copy()
        if not right.empty:
            right_sorted = right.iloc[(right["x_plot"] - C_x).abs().argsort()]
            bottom_right_near = right_sorted.iloc[0]
            bottoms_near.append(bottom_right_near.to_dict())
            if len(right_sorted) > 1:
                bottoms_far.append(right_sorted.iloc[1].to_dict())

    fig, ax = plt.subplots(figsize=(12, 5))

    ax.plot(prof_a_x, prof_a["elevation"], "k-", label=f"Profile {transect_a}")
    ax.plot(prof_b_x, prof_b["elevation"], "k--", label=f"Profile {transect_b}")

    # TOP (blue)
    for top in tops_to_plot:
        ax.scatter(top["x_plot"], top["elevation"], s=70, color="blue", edgecolor="k", zorder=5)
        ax.text(
            top["x_plot"], top["elevation"] + 0.4,
            f"Top\n{top['elevation']:.2f} m",
            ha="center", va="bottom", fontsize=8
        )


    # CENTER (red)
    ax.scatter(C_x, C_z, s=70, color="red", edgecolor="k", zorder=5)
    ax.text(
        C_x, C_z + 0.4,
        f"Center\n{C_z:.2f} m",
        ha="center", va="bottom", fontsize=8
    )

    # BOTTOM near (yellow)
    for b in bottoms_near:
        ax.scatter(b["x_plot"], b["elevation"], s=70, color="yellow", edgecolor="k", zorder=5)
        ax.text(
            b["x_plot"], b["elevation"] - 0.4,
            f"Bottom (near)\n{b['elevation']:.2f} m",
            ha="center", va="top", fontsize=8
        )

    # BOTTOM far (green)
    for b in bottoms_far:
        ax.scatter(b["x_plot"], b["elevation"], s=70, color="green", edgecolor="k", zorder=5)
        ax.text(
            b["x_plot"], b["elevation"] - 0.4,
            f"Bottom (far)\n{b['elevation']:.2f} m",
            ha="center", va="top", fontsize=8
        )

    ax.set_xlabel("Distance (m)")
    ax.set_ylabel("Elevation (m)")
    ax.set_title(f"Cross section – cone {cone_id}  (axis ≈ {angle_axis:.0f}°)")

    # --- LEGEND  ---
    from matplotlib.lines import Line2D
    legend_elements = [
        Line2D([0], [0], marker="o", color="w", markerfacecolor="blue", markeredgecolor="k",
               markersize=8, label="Top"),
        Line2D([0], [0], marker="o", color="w", markerfacecolor="red", markeredgecolor="k",
               markersize=8, label="Center"),
        Line2D([0], [0], marker="o", color="w", markerfacecolor="yellow", markeredgecolor="k",
               markersize=8, label="Bottom (near)"),
        Line2D([0], [0], marker="o", color="w", markerfacecolor="green", markeredgecolor="k",
               markersize=8, label="Bottom (far)"),
        Line2D([0], [0], color="k", linestyle="-", label=f"Profile {transect_a}"),
        Line2D([0], [0], color="k", linestyle="--", label=f"Profile {transect_b}"),
    ]
    ax.legend(handles=legend_elements, loc="upper right")

    # --- CALCULATION OF PARAMETERS Wco, Wcr, H_left/right, D_left/right ---
    metrics_lines = []

    if (top_left is not None) and (bottom_left_near is not None):
        H_left = float(top_left["elevation"] - bottom_left_near["elevation"])
        D_left = float(top_left["elevation"] - C_z)
        metrics_lines.append(f"H_left = {H_left:.2f} m")
        metrics_lines.append(f"D_left = {D_left:.2f} m")
    else:
        H_left = D_left = None

    if (top_right is not None) and (bottom_right_near is not None):
        H_right = float(top_right["elevation"] - bottom_right_near["elevation"])
        D_right = float(top_right["elevation"] - C_z)
        metrics_lines.append(f"H_right = {H_right:.2f} m")
        metrics_lines.append(f"D_right = {D_right:.2f} m")
    else:
        H_right = D_right = None

    if (bottom_left_near is not None) and (bottom_right_near is not None):
        Wco = float(bottom_right_near["x_plot"] - bottom_left_near["x_plot"])
        metrics_lines.insert(0, f"Wco = {Wco:.2f} m")
    else:
        Wco = None

    if (top_left is not None) and (top_right is not None):
        Wcr = float(top_right["x_plot"] - top_left["x_plot"])
        metrics_lines.insert(1, f"Wcr = {Wcr:.2f} m")
    else:
        Wcr = None

    if metrics_lines:
        text_str = "\n".join(metrics_lines)
    else:
        text_str = "No metrics\n(too few points)"

    ax.text(
        0.02, 0.98,
        text_str,
        transform=ax.transAxes,
        ha="left", va="top",
        fontsize=9,
        bbox=dict(boxstyle="round", facecolor="white", alpha=0.8)
    )

    ax.grid(True, alpha=0.3)
    plt.tight_layout()

    out_name = f"cone{cone_id}_axis{int(round(angle_axis))}.svg"
    out_path = os.path.join(output_dir, out_name)
    fig.savefig(out_path, dpi=300)
    plt.close(fig)

    print(f"✔ Saved: {out_path}")



for cone_id in selected_cone_ids:
    cone_id_str = str(cone_id)

    sub = transects_info[transects_info["cone_id"].astype(str) == cone_id_str]
    if sub.empty:
        print(f"[cone {cone_id_str}] no transects – skipped.")
        continue

    tr_info = sub.set_index("transect_id")["angle_deg"].to_dict()

    visited = set()

    print(f"\n=== Cone {cone_id_str} ===")
    for tr_id, ang in tr_info.items():
        if tr_id in visited:
            continue

        base_ang = ang % 360.0
        complement = (base_ang + 180.0) % 360.0

        # looking for a transect whose angle is closest to complementary
        candidate = None
        best_diff = 999.0
        for tr2, ang2 in tr_info.items():
            if tr2 == tr_id:
                continue
            diff = min(abs(ang2 - complement),
                       360.0 - abs(ang2 - complement))
            if diff < best_diff:
                best_diff = diff
                candidate = tr2

        if candidate is None or best_diff > 1e-3:
            # no pair (e.g. only one transect on this axis) – skipped
            continue

        visited.add(tr_id)
        visited.add(candidate)

        axis_mid = (base_ang % 180.0)  # axis angle – e.g.. 90° i 270° -> 90
        print(f"  Axis ~{axis_mid:.1f}° -> pair of opposite transects: {tr_id}, {candidate}")

        # arbitrarily choose which will be "A" and which will be "B"
        plot_axis_pair(cone_id_str, tr_id, candidate, axis_mid)

print("✅ Done – all cross-sections generated.")



=== Cone 55 ===
  Axis ~0.0° -> pair of opposite transects: 55_0deg, 55_180deg
✔ Saved: data/test_set/output/figures/test_set/cross_sections/cone55_axis0.svg
  Axis ~135.0° -> pair of opposite transects: 55_135deg, 55_315deg
✔ Saved: data/test_set/output/figures/test_set/cross_sections/cone55_axis135.svg
  Axis ~45.0° -> pair of opposite transects: 55_225deg, 55_45deg
✔ Saved: data/test_set/output/figures/test_set/cross_sections/cone55_axis45.svg
  Axis ~90.0° -> pair of opposite transects: 55_270deg, 55_90deg


✔ Saved: data/test_set/output/figures/test_set/cross_sections/cone55_axis90.svg

=== Cone 38 ===
  Axis ~0.0° -> pair of opposite transects: 38_0deg, 38_180deg
✔ Saved: data/test_set/output/figures/test_set/cross_sections/cone38_axis0.svg
  Axis ~135.0° -> pair of opposite transects: 38_135deg, 38_315deg
✔ Saved: data/test_set/output/figures/test_set/cross_sections/cone38_axis135.svg
  Axis ~45.0° -> pair of opposite transects: 38_225deg, 38_45deg


✔ Saved: data/test_set/output/figures/test_set/cross_sections/cone38_axis45.svg
  Axis ~90.0° -> pair of opposite transects: 38_270deg, 38_90deg
✔ Saved: data/test_set/output/figures/test_set/cross_sections/cone38_axis90.svg

=== Cone 14 ===
  Axis ~0.0° -> pair of opposite transects: 14_0deg, 14_180deg
✔ Saved: data/test_set/output/figures/test_set/cross_sections/cone14_axis0.svg
  Axis ~135.0° -> pair of opposite transects: 14_135deg, 14_315deg
✔ Saved: data/test_set/output/figures/test_set/cross_sections/cone14_axis135.svg
  Axis ~45.0° -> pair of opposite transects: 14_225deg, 14_45deg


✔ Saved: data/test_set/output/figures/test_set/cross_sections/cone14_axis45.svg
  Axis ~90.0° -> pair of opposite transects: 14_270deg, 14_90deg
✔ Saved: data/test_set/output/figures/test_set/cross_sections/cone14_axis90.svg

=== Cone 22 ===
  Axis ~0.0° -> pair of opposite transects: 22_0deg, 22_180deg
✔ Saved: data/test_set/output/figures/test_set/cross_sections/cone22_axis0.svg
  Axis ~135.0° -> pair of opposite transects: 22_135deg, 22_315deg
✔ Saved: data/test_set/output/figures/test_set/cross_sections/cone22_axis135.svg
  Axis ~45.0° -> pair of opposite transects: 22_225deg, 22_45deg


✔ Saved: data/test_set/output/figures/test_set/cross_sections/cone22_axis45.svg
  Axis ~90.0° -> pair of opposite transects: 22_270deg, 22_90deg
✔ Saved: data/test_set/output/figures/test_set/cross_sections/cone22_axis90.svg

=== Cone 18 ===
  Axis ~0.0° -> pair of opposite transects: 18_0deg, 18_180deg
✔ Saved: data/test_set/output/figures/test_set/cross_sections/cone18_axis0.svg
  Axis ~135.0° -> pair of opposite transects: 18_135deg, 18_315deg


✔ Saved: data/test_set/output/figures/test_set/cross_sections/cone18_axis135.svg
  Axis ~45.0° -> pair of opposite transects: 18_225deg, 18_45deg
✔ Saved: data/test_set/output/figures/test_set/cross_sections/cone18_axis45.svg
  Axis ~90.0° -> pair of opposite transects: 18_270deg, 18_90deg
✔ Saved: data/test_set/output/figures/test_set/cross_sections/cone18_axis90.svg
✅ Done – all cross-sections generated.
